# Create ADA Awards (American Diabetes Association)

Creates American Diabetes Association research-grant awards from two
first-party ADA publications:
1. **Research Database detail pages** on professional.diabetes.org
   (`/rdb/<slug>`, 161 pages, all enumerated by the site's XML sitemap). The old
   searchable listing (`/research-grants/research-database`) now redirects to
   the Research & Grants landing page, but the detail pages are live. Each has
   project title, PI, institution, **Grant Number**, type of grant (Basic /
   Clinical / Translational), diabetes type, focus areas, project start/end
   dates, status and a research description.
2. **Pathway to Stop Diabetes recipients** (diabetes.org/research/recipients):
   34 Pathway Initiator / Accelerator / Visionary grants 2014-2020 with name,
   institution, title, Grant #; 8 are also in the research database (the
   research-database row wins), 26 are added.

**187 grants, award years 2014-2024** (local run 2026-09-30). ADA reports
230+ active grants; the public detail pages cover 161 of its portfolio (active
and recently completed), so this is a partial but first-party list.

**No amounts:** ADA does not publish per-grant amounts, so `amount` /
`currency` are NULL (Step 6.7 amount check waived; noted in the tracker).

**PI names:** the visible name block drops the surname for some PIs
("Angela  G, PhD"); the script takes the full name from the page's Drupal
datalayer taxonomy term (`"investigator":{"556":"Angela G Brega"}`) and keeps the
visible string in `lead_name_display`.

**`funder_award_id` (section 2.1.1):** the printed ADA Grant Number
(`1-18-INI-14`, `11-23-PDF-04`, `7-23-ICTSWH-08`, and `CDTR-NN` for the
Diabetes Disparities/CDTR program), which is exactly the form citing works
write in acknowledgements (`crossref_work.grants` / `europepmc_work_funders`
shells: `1-18-IBS-233`, `7-21-JDF-020`, ...). ADA registers no grant DOIs.
`start_year` falls back to the award year encoded in the grant number
(`1-18-...` -> 2018) for the Pathway rows that have no project dates.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306089`
- display_name: American Diabetes Association
- ror_id / doi: from `openalex.funders.funders` (ror 04f6cgz95, doi 10.13039/100000041)
- Not `4320306226` (American Diabetes Association Research Foundation), a
  separate OpenAlex funder row with far fewer works; grants are awarded by ADA.

**Prerequisites:** run `scripts/local/ada_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/ada/ada_projects.parquet` (section 1.4 shrink guard).

**Priority:** `496` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ada_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ada/ada_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_award_ids FROM openalex.awards.ada_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.ada_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ada_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306089 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306089;

## Step 2: Create ADA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ada_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306089  -- American Diabetes Association
),

g AS (
    SELECT
        *,
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.ada_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,      -- ADA publishes no per-grant amounts
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.grant_type), '') as funder_scheme,
    'ada_research_database' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), TRY_CAST(g.grant_year AS INT)) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is an ADA grant number
-- ("1-18-INI-14", "11-23-ICTSWH-08", "4-23-CVD1-01") or a CDTR-NN program number.
SELECT assert_true(
    COUNT_IF(NOT (funder_award_id RLIKE '^[0-9]{1,2}-[0-9]{2}-[A-Z0-9]+-[0-9]+$' OR funder_award_id RLIKE '^CDTR-[0-9]+$')) = 0
    AND COUNT(*) = COUNT(DISTINCT id),
    'ada_awards: unexpected funder_award_id shape or duplicate ids'
) AS shape_ok
FROM openalex.awards.ada_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ada_research_database' AND priority = 496;

-- Insert into openalex_awards_raw with priority.
-- Priority 496: direct-from-funder ingest of ADA's own research database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 496 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    496 as priority
FROM openalex.awards.ada_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_ada_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.ada_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, doi
FROM openalex.awards.ada_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, COUNT(lead_investigator) as n_pi
FROM openalex.awards.ada_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, COUNT(lead_investigator) as n_pi
FROM openalex.awards.ada_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.ada_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.ada_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(doi) as has_doi,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.ada_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement shells (priority 0) that cite the same
-- ADA grant number: these works attach to our awards in CreateAwards dedup.
SELECT c.provenance, COUNT(*) as shell_rows, COUNT(t.id) as matched_to_ada_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.ada_awards t ON t.id = c.id
WHERE c.funder_id = 4320306089 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ada_research_database'
GROUP BY provenance, priority;